In [1]:
from pyspark.sql import SparkSession

# Créer une session Spark
spark = SparkSession.builder.appName("TestPySpark").getOrCreate()

# Créer un DataFrame simple
data = [("Alice", 1), ("Bob", 2), ("Cathy", 3)]
df = spark.createDataFrame(data, ["Name", "Age"])

# Afficher le DataFrame
df.show()

# Fermer la session Spark  
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/02 15:25:36 WARN Utils: Your hostname, DESKTOP-86IRBA9, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/02 15:25:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 15:25:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/02 15:25:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+-----+---+
| Name|Age|
+-----+---+
|Alice|  1|
|  Bob|  2|
|Cathy|  3|
+-----+---+



In [2]:
import pandas as pd

df = pd.read_parquet("ticket_management/output/tickets/")
print(df.head(10))
print(f"\nNombre total de tickets exportés : {len(df)}")

   ticket_id client_id                 created_at  \
0  TKT-00042   CLI-009 2026-07-03 10:53:23.516087   
1  TKT-00010   CLI-014 2026-07-03 10:50:43.488760   
2  TKT-00011   CLI-039 2026-07-03 10:50:48.490578   
3  TKT-00052   CLI-014 2026-07-03 10:54:13.529339   
4  TKT-00014   CLI-004 2026-07-03 10:51:03.492814   
5  TKT-00015   CLI-039 2026-07-03 10:51:08.493361   
6  TKT-00002   CLI-034 2026-07-03 10:50:03.481209   
7  TKT-00021   CLI-011 2026-07-03 10:51:38.498893   
8  TKT-00032   CLI-031 2026-07-03 10:52:33.507895   
9  TKT-00025   CLI-038 2026-07-03 10:51:58.502475   

                                          demande           type_demande  \
0     Je n'arrive pas à me connecter à mon compte            Performance   
1            Les temps de réponse sont très lents     Incident technique   
2  Besoin d'une extraction de données historiques            Performance   
3     Je n'arrive pas à me connecter à mon compte     Incident technique   
4  Besoin d'une extraction de donnée

In [6]:
df_agg = pd.read_parquet("ticket_management/output/agregations_priorite/")
# Convertir les timestamps nanoseconde en dates lisibles
df_agg['window_start'] = pd.to_datetime(df_agg['window'].apply(lambda x: x['start']))
df_agg['window_end'] = pd.to_datetime(df_agg['window'].apply(lambda x: x['end']))
df_agg = df_agg.drop(columns=['window'])

# Trier par fenêtre puis priorité
df_agg = df_agg.sort_values(['window_start', 'priorite'])

print(df_agg.to_string(index=False))

priorite  nb_tickets        window_start          window_end
Critique           1 2026-07-03 10:49:00 2026-07-03 10:50:00
   Basse           2 2026-07-03 10:50:00 2026-07-03 10:51:00
Critique           5 2026-07-03 10:50:00 2026-07-03 10:51:00
   Haute           3 2026-07-03 10:50:00 2026-07-03 10:51:00
 Moyenne           2 2026-07-03 10:50:00 2026-07-03 10:51:00
   Basse           3 2026-07-03 10:51:00 2026-07-03 10:52:00
Critique           1 2026-07-03 10:51:00 2026-07-03 10:52:00
   Haute           4 2026-07-03 10:51:00 2026-07-03 10:52:00
 Moyenne           4 2026-07-03 10:51:00 2026-07-03 10:52:00
   Basse           5 2026-07-03 10:52:00 2026-07-03 10:53:00
Critique           3 2026-07-03 10:52:00 2026-07-03 10:53:00
   Haute           2 2026-07-03 10:52:00 2026-07-03 10:53:00
 Moyenne           2 2026-07-03 10:52:00 2026-07-03 10:53:00
   Basse           3 2026-07-03 10:53:00 2026-07-03 10:54:00
Critique           2 2026-07-03 10:53:00 2026-07-03 10:54:00
   Haute           4 202